In [4]:
# IMPORTAÇÕES
import os
import pandas as pd
import numpy as np
from pathlib import Path

# DEFINIÇÃO DE DIRETÓRIOS
# Pasta de resultados contém as 6 pastas temáticas
RESULTADOS_DIR = Path('./resultados')
TABELAS_DIR = Path('./relatorio/tabelas_latex')

# Criar diretório se não existir
TABELAS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Diretórios configurados:")
print(f"   📁 Resultados: {RESULTADOS_DIR.resolve()}")
print(f"   📁 Tabelas LaTeX: {TABELAS_DIR.resolve()}")

# Descobrir todos os CSVs nas pastas temáticas
print(f"\n📂 Procurando arquivos CSV em resultados/...")
csvs_encontrados = list(RESULTADOS_DIR.glob('*/*.csv'))
print(f"✅ {len(csvs_encontrados)} arquivo(s) CSV encontrado(s):\n")

for csv_path in sorted(csvs_encontrados):
    print(f"   • {csv_path.relative_to(RESULTADOS_DIR)}")

✅ Diretórios configurados:
   📁 Resultados: D:\GitHub\DoutoradoCefet\PlanejamentoAnaliseEstatisticoExperimentos\Trabalhos\estudo_dirigido_3\resultados
   📁 Tabelas LaTeX: D:\GitHub\DoutoradoCefet\PlanejamentoAnaliseEstatisticoExperimentos\Trabalhos\estudo_dirigido_3\relatorio\tabelas_latex

📂 Procurando arquivos CSV em resultados/...
✅ 16 arquivo(s) CSV encontrado(s):

   • 01_Parametros_Metodologia\Parametros_Analise.csv
   • 02_Estatisticas_Descritivas\Estatisticas_Descritivas_Gerais.csv
   • 03_Teorema_Central_Limite\TCL_Resumo_Comparativo.csv
   • 03_Teorema_Central_Limite\TCL_Validacao_10mil_simulacoes.csv
   • 04_Intervalos_Confianca\IC_95_Por_Ano_Detalhado.csv
   • 04_Intervalos_Confianca\IC_95_Por_Mes_Detalhado.csv
   • 05_ANOVA_Fatorial\ANOVA_Efeitos_Principais_Interacao.csv
   • 05_ANOVA_Fatorial\ANOVA_Qualidade_Modelo.csv
   • 06_Conclusoes_Relatorio\Conclusoes_Finais_Interpretacao.csv
   • informacoes_dados\bd_gta_dentro_mg202505091607_analise.csv
   • informacoes_dados\bd_

## Geração Automática de Tabelas LaTeX

Lendo todos os CSVs das pastas temáticas e convertendo para LaTeX

In [5]:
# FUNÇÃO AUXILIAR PARA GERAR TABELAS LaTeX
def dataframe_to_latex_table(df, titulo, label, pasta_origem="", nome_original=""):
    """
    Converte um DataFrame em uma tabela LaTeX formatada
    
    Args:
        df: DataFrame com os dados
        titulo: Título da tabela
        label: Label para referência (ex: tab:dados_1)
        pasta_origem: Nome da pasta de origem dos dados
        nome_original: Nome do arquivo CSV original
    """
    
    # Determinar número de colunas e ajustar formato
    ncols = len(df.columns)
    
    # Criar especificação de colunas (usar 'l' para texto, 'r' para números)
    col_spec = '|' + '|'.join(['l' if i == 0 else 'r' for i in range(ncols)]) + '|'
    
    # Header da tabela
    tex = r'\begin{table}[H]' + '\n'
    tex += r'\centering' + '\n'
    tex += r'\small' + '\n'
    tex += f'\\caption{{{titulo}}}' + '\n'
    tex += f'\\label{{{label}}}' + '\n'
    tex += f'\\begin{{tabular}}{{{col_spec}}}' + '\n'
    tex += r'\hline' + '\n'
    
    # Cabeçalho (nomes das colunas)
    header = ' & '.join([f'\\textbf{{{col}}}' for col in df.columns])
    tex += header + r' \\' + '\n'
    tex += r'\hline' + '\n'
    
    # Dados
    for _, row in df.iterrows():
        valores = []
        for val in row:
            # Formatar números
            if isinstance(val, (int, np.integer)):
                valores.append(str(int(val)))
            elif isinstance(val, (float, np.floating)):
                # Formatar com 4 casas decimais se for pequeno, 2 se for grande
                if abs(val) < 1000:
                    valores.append(f'{val:.4f}')
                else:
                    valores.append(f'{val:.2f}')
            else:
                valores.append(str(val))
        
        tex += ' & '.join(valores) + r' \\' + '\n'
    
    tex += r'\hline' + '\n'
    
    # Nota com origem dos dados
    if pasta_origem and nome_original:
        nota = f'Fonte: \\textit{{{pasta_origem}/{nome_original}}}'
    else:
        nota = ''
    
    tex += r'\end{tabular}' + '\n'
    if nota:
        tex += r'\begin{flushleft}' + '\n'
        tex += f'\\small {nota}' + '\n'
        tex += r'\end{flushleft}' + '\n'
    tex += r'\end{table}' + '\n'
    
    return tex


# GERAR TABELAS PARA TODOS OS CSVs
print('='*80)
print('GERANDO TABELAS LaTeX PARA TODOS OS CSVs')
print('='*80)

# Descobrir todos os CSVs
csvs = sorted(RESULTADOS_DIR.glob('*/*.csv'))

tabelas_info = []
contador = 1

for csv_path in csvs:
    pasta_nome = csv_path.parent.name
    arquivo_nome = csv_path.name
    
    try:
        # Ler o CSV
        df = pd.read_csv(csv_path, encoding='utf-8')
        
        # Criar título limpo a partir do nome do arquivo
        titulo_limpo = arquivo_nome.replace('.csv', '').replace('_', ' ').title()
        label = f"tab:{contador:02d}_{pasta_nome}"
        
        # Gerar tabela LaTeX
        tex_table = dataframe_to_latex_table(
            df, 
            titulo_limpo, 
            label,
            pasta_nome,
            arquivo_nome
        )
        
        # Salvar em arquivo separado
        tex_filename = f"tab_{contador:02d}_{arquivo_nome.replace('.csv', '.tex')}"
        tex_filepath = TABELAS_DIR / tex_filename
        
        with open(tex_filepath, 'w', encoding='utf-8') as f:
            f.write(tex_table)
        
        tabelas_info.append({
            'numero': contador,
            'arquivo': arquivo_nome,
            'pasta': pasta_nome,
            'arquivo_tex': tex_filename,
            'linhas': len(df),
            'colunas': len(df.columns)
        })
        
        print(f"\n✅ Tabela {contador:02d}: {arquivo_nome}")
        print(f"   📂 Pasta: {pasta_nome}")
        print(f"   📊 Dados: {len(df)} linhas × {len(df.columns)} colunas")
        print(f"   📄 Salvo em: {tex_filename}")
        
        contador += 1
        
    except Exception as e:
        print(f"\n❌ Erro ao processar {csv_path}:")
        print(f"   {str(e)}")

print('\n' + '='*80)
print('RESUMO DAS TABELAS GERADAS')
print('='*80)

# Exibir resumo
df_resumo = pd.DataFrame(tabelas_info)
print(f"\n{df_resumo.to_string(index=False)}")
print(f"\nTotal: {len(tabelas_info)} tabelas geradas")
print(f"Localização: {TABELAS_DIR.resolve()}")

GERANDO TABELAS LaTeX PARA TODOS OS CSVs

✅ Tabela 01: Parametros_Analise.csv
   📂 Pasta: 01_Parametros_Metodologia
   📊 Dados: 5 linhas × 2 colunas
   📄 Salvo em: tab_01_Parametros_Analise.tex

✅ Tabela 02: Estatisticas_Descritivas_Gerais.csv
   📂 Pasta: 02_Estatisticas_Descritivas
   📊 Dados: 6 linhas × 2 colunas
   📄 Salvo em: tab_02_Estatisticas_Descritivas_Gerais.tex

✅ Tabela 03: TCL_Resumo_Comparativo.csv
   📂 Pasta: 03_Teorema_Central_Limite
   📊 Dados: 5 linhas × 2 colunas
   📄 Salvo em: tab_03_TCL_Resumo_Comparativo.tex

✅ Tabela 04: TCL_Validacao_10mil_simulacoes.csv
   📂 Pasta: 03_Teorema_Central_Limite
   📊 Dados: 4 linhas × 3 colunas
   📄 Salvo em: tab_04_TCL_Validacao_10mil_simulacoes.tex

✅ Tabela 05: IC_95_Por_Ano_Detalhado.csv
   📂 Pasta: 04_Intervalos_Confianca
   📊 Dados: 15 linhas × 8 colunas
   📄 Salvo em: tab_05_IC_95_Por_Ano_Detalhado.tex

✅ Tabela 06: IC_95_Por_Mes_Detalhado.csv
   📂 Pasta: 04_Intervalos_Confianca
   📊 Dados: 12 linhas × 8 colunas
   📄 Salvo em

In [6]:
# CRIAR ARQUIVO DE ÍNDICE PARA FACILITAR USO NO RELATÓRIO

print('\n' + '='*80)
print('CRIANDO ARQUIVO DE ÍNDICE')
print('='*80)

# Encontrar todos os arquivos .tex gerados
tex_files = sorted(TABELAS_DIR.glob('tab_*.tex'))

# Criar conteúdo do índice
indice_content = r"""%% ÍNDICE DE TABELAS LaTeX
%% Arquivo gerado automaticamente
%% Data: """ + pd.Timestamp.now().strftime('%d/%m/%Y %H:%M:%S') + r"""

\section{Tabelas de Resultados}

Nesta seção são apresentadas as tabelas com os resultados das análises estatísticas.

"""

# Adicionar entrada para cada tabela
indice_content += r"""
% Para incluir cada tabela no relatório, use:
% \input{relatorio/tabelas_latex/tab_XX_nome_tabela.tex}

"""

indice_content += "% TABELAS DISPONÍVEIS:\n"

for i, tex_file in enumerate(tex_files, 1):
    # Extrair número e nome da tabela
    nome_arquivo = tex_file.stem
    numero = nome_arquivo.split('_')[1]
    resto = nome_arquivo.replace(f'tab_{numero}_', '')
    
    indice_content += f"% {i}. \\input{{relatorio/tabelas_latex/{tex_file.name}}} % {resto}\n"

# Salvar o arquivo de índice
indice_file = TABELAS_DIR / 'INDICE_TABELAS.txt'
with open(indice_file, 'w', encoding='utf-8') as f:
    f.write(indice_content)

print(f"\n✅ Arquivo de índice criado: {indice_file.name}")
print(f"   Localização: {indice_file}")

# Listar todas as tabelas com comandos de inclusão prontos
print("\n" + '='*80)
print('COMANDOS PARA INCLUIR TABELAS NO RELATÓRIO LaTeX')
print('='*80)
print("\nCopie e cole os comandos abaixo onde deseja incluir as tabelas:\n")

for i, tex_file in enumerate(tex_files, 1):
    print(f"% Tabela {i}: {tex_file.stem}")
    print(f"\\input{{relatorio/tabelas_latex/{tex_file.name}}}\n")

print("\n" + '='*80)
print('✅ TABELAS GERADAS COM SUCESSO!')
print('='*80)
print(f"\nTotal de tabelas: {len(tex_files)}")
print(f"Localização: {TABELAS_DIR.resolve()}")
print(f"\nDica: Use \\input{{relatorio/tabelas_latex/tab_XX_nome.tex}} no seu arquivo .tex principal")


CRIANDO ARQUIVO DE ÍNDICE

✅ Arquivo de índice criado: INDICE_TABELAS.txt
   Localização: relatorio\tabelas_latex\INDICE_TABELAS.txt

COMANDOS PARA INCLUIR TABELAS NO RELATÓRIO LaTeX

Copie e cole os comandos abaixo onde deseja incluir as tabelas:

% Tabela 1: tab_01_Parametros_Analise
\input{relatorio/tabelas_latex/tab_01_Parametros_Analise.tex}

% Tabela 2: tab_02_Estatisticas_Descritivas_Gerais
\input{relatorio/tabelas_latex/tab_02_Estatisticas_Descritivas_Gerais.tex}

% Tabela 3: tab_03_TCL_Resumo_Comparativo
\input{relatorio/tabelas_latex/tab_03_TCL_Resumo_Comparativo.tex}

% Tabela 4: tab_04_TCL_Validacao_10mil_simulacoes
\input{relatorio/tabelas_latex/tab_04_TCL_Validacao_10mil_simulacoes.tex}

% Tabela 5: tab_05_IC_95_Por_Ano_Detalhado
\input{relatorio/tabelas_latex/tab_05_IC_95_Por_Ano_Detalhado.tex}

% Tabela 6: tab_06_IC_95_Por_Mes_Detalhado
\input{relatorio/tabelas_latex/tab_06_IC_95_Por_Mes_Detalhado.tex}

% Tabela 7: tab_07_ANOVA_Efeitos_Principais_Interacao
\input{relat

## Verificação Final - Resumo das Tabelas Geradas

In [7]:
# VERIFICAÇÃO FINAL
print("="*80)
print("VERIFICAÇÃO FINAL - RESUMO COMPLETO")
print("="*80)

# Verificar se todos os arquivos foram criados
tex_files = sorted(TABELAS_DIR.glob('tab_*.tex'))

print(f"\n📊 TABELAS GERADAS: {len(tex_files)}")
print(f"📁 Localização: {TABELAS_DIR.resolve()}\n")

# Agrupar por pasta temática
tabelas_por_grupo = {}
for tex_file in tex_files:
    # Ler o arquivo para extrair informações
    with open(tex_file, 'r', encoding='utf-8') as f:
        conteudo = f.read()
    
    # Extrair informações
    linhas = conteudo.count(r'\\')
    bytes_size = os.path.getsize(tex_file)
    
    # Determinar o grupo baseado no número
    numero = int(tex_file.stem.split('_')[1])
    
    if numero <= 2:
        grupo = "01_Parametros_Metodologia"
    elif numero <= 3:
        grupo = "02_Estatisticas_Descritivas"
    elif numero <= 4:
        grupo = "03_Teorema_Central_Limite"
    elif numero <= 6:
        grupo = "04_Intervalos_Confianca"
    elif numero <= 8:
        grupo = "05_ANOVA_Fatorial"
    elif numero == 9:
        grupo = "06_Conclusoes_Relatorio"
    else:
        grupo = "Dados Adicionais"
    
    if grupo not in tabelas_por_grupo:
        tabelas_por_grupo[grupo] = []
    
    tabelas_por_grupo[grupo].append({
        'nome': tex_file.name,
        'linhas': linhas,
        'bytes': bytes_size
    })

# Exibir resultado
for grupo, tabelas in sorted(tabelas_por_grupo.items()):
    print(f"✅ {grupo}/ ({len(tabelas)} tabela(s))")
    for tab in tabelas:
        print(f"   • {tab['nome']} ({tab['bytes']} bytes)")

print("\n" + "="*80)
print("✅ PROCESSO CONCLUÍDO COM SUCESSO!")
print("="*80)
print(f"\n📌 Próximas etapas:")
print(f"   1. Use \\input{{relatorio/tabelas_latex/tab_XX_nome.tex}} no seu arquivo .tex")
print(f"   2. Todas as tabelas estão prontas para serem incorporadas ao relatório")
print(f"   3. Consulte o arquivo INDICE_TABELAS.txt para a lista completa")

VERIFICAÇÃO FINAL - RESUMO COMPLETO

📊 TABELAS GERADAS: 16
📁 Localização: D:\GitHub\DoutoradoCefet\PlanejamentoAnaliseEstatisticoExperimentos\Trabalhos\estudo_dirigido_3\relatorio\tabelas_latex

✅ 01_Parametros_Metodologia/ (2 tabela(s))
   • tab_01_Parametros_Analise.tex (517 bytes)
   • tab_02_Estatisticas_Descritivas_Gerais.tex (509 bytes)
✅ 02_Estatisticas_Descritivas/ (1 tabela(s))
   • tab_03_TCL_Resumo_Comparativo.tex (539 bytes)
✅ 03_Teorema_Central_Limite/ (1 tabela(s))
   • tab_04_TCL_Validacao_10mil_simulacoes.tex (597 bytes)
✅ 04_Intervalos_Confianca/ (2 tabela(s))
   • tab_05_IC_95_Por_Ano_Detalhado.tex (1663 bytes)
   • tab_06_IC_95_Por_Mes_Detalhado.tex (1399 bytes)
✅ 05_ANOVA_Fatorial/ (2 tabela(s))
   • tab_07_ANOVA_Efeitos_Principais_Interacao.tex (648 bytes)
   • tab_08_ANOVA_Qualidade_Modelo.tex (538 bytes)
✅ 06_Conclusoes_Relatorio/ (1 tabela(s))
   • tab_09_Conclusoes_Finais_Interpretacao.tex (643 bytes)
✅ Dados Adicionais/ (7 tabela(s))
   • tab_10_bd_gta_dentro_